# Tomahawk Revision 2.2 — R-MAPLA Proof-of-Concept Kaggle Test

Run all cells. The notebook validates the updated 16-port dual-ring laser geometry and pulse bookkeeping, runs a deterministic 0-D D-T power screen, performs a confinement-time robustness test, and then executes smoke and full Monte Carlo proof-of-concept screens automatically.


In [1]:
# CELL 1 — Kaggle-safe settings and dependency check
import json
import math
import os
import platform
import time
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

os.environ.setdefault("MPLCONFIGDIR", "/tmp/tomahawk-rev22-poc-mpl")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)

SEED = int(os.environ.get("TOMAHAWK_POC_SEED", "20260825"))
SMOKE_SAMPLES = int(os.environ.get("TOMAHAWK_POC_SMOKE_SAMPLES", "8192"))
FULL_SAMPLES = int(os.environ.get("TOMAHAWK_POC_FULL_SAMPLES", str(2**18)))
MAX_HOURS = float(os.environ.get("TOMAHAWK_MAX_HOURS", "9.25"))
OUTPUT_ROOT = Path(
    os.environ.get(
        "TOMAHAWK_POC_OUTPUT_ROOT",
        "/kaggle/working/tomahawk_rev22_poc" if Path("/kaggle/working").exists() else str(Path.cwd() / "tomahawk_rev22_poc"),
    )
).expanduser().resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

required = ("numpy", "pandas", "matplotlib")
print("Kaggle proof-of-concept environment:", {name: True for name in required})
print("Seed:", SEED)
print("Smoke cases:", SMOKE_SAMPLES, "Full cases:", FULL_SAMPLES)
print("Output root:", OUTPUT_ROOT)
print("No SciPy, scikit-learn, custom package, or network access is required.")


Kaggle proof-of-concept environment: {'numpy': True, 'pandas': True, 'matplotlib': True}
Seed: 20260825
Smoke cases: 8192 Full cases: 262144
Output root: /kaggle/working/tomahawk_rev22_poc
No SciPy, scikit-learn, custom package, or network access is required.


## Updated laser-array geometry


In [2]:
# CELL 2 — Updated R-MAPLA 16-port dual-ring geometry
RMAPLA_PORTS = 16
RMAPLA_RING_COUNT = 2
RMAPLA_PORTS_PER_RING = 8
RMAPLA_TOROIDAL_STEP_DEG = 22.5
RMAPLA_POLOIDAL_ANGLES_DEG = np.array(
    [-60.0, -45.0, -30.0, -15.0, 15.0, 30.0, 45.0, 60.0], dtype=np.float64
)
RMAPLA_WAVELENGTH_NM = 351.0
RMAPLA_PULSE_ENERGY_J = 2.50e6
RMAPLA_REPETITION_HZ = 10.0
RMAPLA_PREPULSE_S = 5.0e-9
RMAPLA_COMPRESSION_SPIKE_S = 10.0e-12
RMAPLA_FOCAL_OFFSET_ENVELOPE_M = 0.050  # provisional symmetric optical envelope
R0_M = 7.5
A_M = 3.0
FOCUS = np.array([0.0, 0.0, 0.0], dtype=np.float64)


def build_rmapla_ports(major_radius_m=R0_M, minor_radius_m=A_M):
    rows = []
    for sequence in range(RMAPLA_PORTS):
        ring = (sequence % RMAPLA_RING_COUNT) + 1
        poloidal_index = sequence // RMAPLA_RING_COUNT
        phi_deg = sequence * RMAPLA_TOROIDAL_STEP_DEG
        theta_deg = float(RMAPLA_POLOIDAL_ANGLES_DEG[poloidal_index])
        phi = np.deg2rad(phi_deg)
        theta = np.deg2rad(theta_deg)
        r_minor = 0.92 * minor_radius_m
        position = np.array(
            [
                (major_radius_m + r_minor * np.cos(theta)) * np.cos(phi),
                (major_radius_m + r_minor * np.cos(theta)) * np.sin(phi),
                r_minor * np.sin(theta),
            ],
            dtype=np.float64,
        )
        focal_offset_z_m = RMAPLA_FOCAL_OFFSET_ENVELOPE_M * np.sin(theta)
        focal_point = FOCUS + np.array([0.0, 0.0, focal_offset_z_m])
        direction = focal_point - position
        direction /= np.linalg.norm(direction)
        rows.append(
            {
                "port_id": f"RMAPLA-{sequence + 1:02d}",
                "sequence": sequence + 1,
                "ring": ring,
                "toroidal_phase_deg": phi_deg,
                "poloidal_angle_deg": theta_deg,
                "x_m": position[0],
                "y_m": position[1],
                "z_m": position[2],
                "focus_x_m": focal_point[0],
                "focus_y_m": focal_point[1],
                "focus_z_m": focal_point[2],
                "dir_x": direction[0],
                "dir_y": direction[1],
                "dir_z": direction[2],
                "focal_offset_z_m": focal_offset_z_m,
            }
        )
    return pd.DataFrame(rows)


def validate_rmapla_ports(ports):
    phases = np.sort(ports["toroidal_phase_deg"].to_numpy(dtype=np.float64))
    steps = np.diff(np.r_[phases, phases[0] + 360.0])
    dirs = ports[["dir_x", "dir_y", "dir_z"]].to_numpy(dtype=np.float64)
    norms = np.linalg.norm(dirs, axis=1)
    checks = {
        "port_count_pass": len(ports) == RMAPLA_PORTS,
        "dual_ring_pass": ports["ring"].nunique() == RMAPLA_RING_COUNT,
        "ports_per_ring_pass": bool((ports.groupby("ring").size() == RMAPLA_PORTS_PER_RING).all()),
        "toroidal_step_pass": bool(np.allclose(steps, RMAPLA_TOROIDAL_STEP_DEG)),
        "poloidal_set_pass": bool(np.array_equal(np.sort(ports["poloidal_angle_deg"].unique()), RMAPLA_POLOIDAL_ANGLES_DEG)),
        "beam_direction_norm_pass": bool(np.allclose(norms, 1.0, atol=1e-12)),
        "phase_step_min_deg": float(steps.min()),
        "phase_step_max_deg": float(steps.max()),
    }
    checks["all_geometry_checks_pass"] = all(value for key, value in checks.items() if key.endswith("_pass"))
    return checks


def plot_rmapla_ports(ports, output_path):
    fig = plt.figure(figsize=(10.5, 8.0), dpi=160)
    ax = fig.add_subplot(111, projection="3d")
    u = np.linspace(0.0, 2.0 * np.pi, 120)
    v = np.linspace(0.0, 2.0 * np.pi, 36)
    uu, vv = np.meshgrid(u, v)
    xx = (R0_M + A_M * np.cos(vv)) * np.cos(uu)
    yy = (R0_M + A_M * np.cos(vv)) * np.sin(uu)
    zz = A_M * np.sin(vv)
    ax.plot_surface(xx, yy, zz, color="#5f6b7a", alpha=0.13, linewidth=0.0, rstride=2, cstride=4)
    colors = {1: "#9a6cff", 2: "#32c7ef"}
    for ring, group in ports.groupby("ring"):
        color = colors[int(ring)]
        ax.scatter(group["x_m"], group["y_m"], group["z_m"], s=42, color=color, label=f"Ring {ring}")
        for row in group.itertuples(index=False):
            ax.plot([row.x_m, row.focus_x_m], [row.y_m, row.focus_y_m], [row.z_m, row.focus_z_m], color=color, alpha=0.60, linewidth=0.9)
    ax.scatter([0.0], [0.0], [0.0], s=90, color="white", edgecolor="#9a6cff", linewidth=1.4, label="Focal region")
    ax.set_title("Tomahawk R-MAPLA: 16-Port Dual-Ring Launch Geometry", pad=18)
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.set_zlabel("z (m)")
    ax.set_box_aspect((1.0, 1.0, 0.55))
    ax.view_init(elev=27.0, azim=-58.0)
    ax.legend(loc="upper left")
    ax.text2D(
        0.02,
        0.02,
        "16 ports | 22.5° toroidal phase step | ±15°, ±30°, ±45°, ±60° poloidal set\n"
        "351 nm | 2.5 MJ/module pulse | 5 ns prepulse + 10 ps spike | 10 Hz",
        transform=ax.transAxes,
        fontsize=8.5,
        color="#164e63",
        bbox={"boxstyle": "round,pad=0.4", "facecolor": "white", "alpha": 0.88, "edgecolor": "#32c7ef"},
    )
    fig.tight_layout()
    fig.savefig(output_path, bbox_inches="tight")
    plt.close(fig)


PORTS = build_rmapla_ports()
GEOMETRY_DIR = OUTPUT_ROOT / "rmapla_geometry"
GEOMETRY_DIR.mkdir(parents=True, exist_ok=True)
GEOMETRY_CHECKS = validate_rmapla_ports(PORTS)
PORTS.to_csv(GEOMETRY_DIR / "rmapla_16_port_geometry.csv", index=False)
(GEOMETRY_DIR / "rmapla_geometry_checks.json").write_text(json.dumps(GEOMETRY_CHECKS, indent=2), encoding="utf-8")
plot_rmapla_ports(PORTS, GEOMETRY_DIR / "rmapla_16_port_geometry.png")
if not GEOMETRY_CHECKS["all_geometry_checks_pass"]:
    raise AssertionError(GEOMETRY_CHECKS)
print("R-MAPLA checks PASS:", GEOMETRY_CHECKS)
print("Geometry outputs:", GEOMETRY_DIR)


R-MAPLA checks PASS: {'port_count_pass': True, 'dual_ring_pass': True, 'ports_per_ring_pass': True, 'toroidal_step_pass': True, 'poloidal_set_pass': True, 'beam_direction_norm_pass': True, 'phase_step_min_deg': 22.5, 'phase_step_max_deg': 22.5, 'all_geometry_checks_pass': True}
Geometry outputs: /kaggle/working/tomahawk_rev22_poc/rmapla_geometry


## Laser pulse and timing checks


In [3]:
# CELL 3 — Laser pulse and timing proof-of-concept checks
per_port_pulse_energy_J = RMAPLA_PULSE_ENERGY_J / RMAPLA_PORTS
laser_average_power_W = RMAPLA_PULSE_ENERGY_J * RMAPLA_REPETITION_HZ
per_port_average_power_W = per_port_pulse_energy_J * RMAPLA_REPETITION_HZ
pulse_checks = {
    "energy_conservation_pass": bool(np.isclose(per_port_pulse_energy_J * RMAPLA_PORTS, RMAPLA_PULSE_ENERGY_J)),
    "positive_timing_pass": bool(RMAPLA_PREPULSE_S > 0.0 and RMAPLA_COMPRESSION_SPIKE_S > 0.0),
    "wavelength_spec_pass": bool(np.isclose(RMAPLA_WAVELENGTH_NM, 351.0)),
    "repetition_spec_pass": bool(np.isclose(RMAPLA_REPETITION_HZ, 10.0)),
    "module_average_power_MW": laser_average_power_W / 1.0e6,
    "per_port_pulse_energy_kJ": per_port_pulse_energy_J / 1.0e3,
    "per_port_average_power_MW": per_port_average_power_W / 1.0e6,
    "prepulse_ns": RMAPLA_PREPULSE_S * 1.0e9,
    "compression_spike_ps": RMAPLA_COMPRESSION_SPIKE_S * 1.0e12,
}
pd.DataFrame([pulse_checks]).to_csv(OUTPUT_ROOT / "laser_pulse_checks.csv", index=False)
print("Laser pulse checks:", pulse_checks)
assert all(pulse_checks[key] for key in pulse_checks if key.endswith("_pass"))


Laser pulse checks: {'energy_conservation_pass': True, 'positive_timing_pass': True, 'wavelength_spec_pass': True, 'repetition_spec_pass': True, 'module_average_power_MW': 25.0, 'per_port_pulse_energy_kJ': 156.25, 'per_port_average_power_MW': 1.5625, 'prepulse_ns': 5.0, 'compression_spike_ps': 10.0}


## Dependency-free D-T physics screen


In [4]:
# CELL 4 — Dependency-free D-T 0-D physics screen
MU0 = 4.0e-7 * math.pi
KB = 1.380649e-23
AMU = 1.66053906660e-27
KEV_TO_J = 1.0e3 * 1.602176634e-19
MEV_TO_J = 1.0e6 * 1.602176634e-19
M_D = 2.01410177812 * AMU
M_T = 3.01604928199 * AMU
E_FUSION = 17.6 * MEV_TO_J
ALPHA_FRACTION = 3.5 / 17.6
NEUTRON_FRACTION = 14.1 / 17.6
CAD_PLASMA_VOLUME_M3 = 950.0
N_D0_BASE = 1.2e20
N_T0_BASE = 1.2e20
CP_ALSB_J_KG_K = 3450.0
ACTIVE_MODULES = 10
FUSION_SOURCE_TARGET_W_PER_MODULE = 5.0e9
CAPTURED_HEAT_TARGET_W_PER_MODULE = 5.0e9
NET_ELECTRIC_TARGET_GWE = 17.0
GROSS_ELECTRIC_TARGET_GWE = 20.0
RECIRCULATING_CEILING_GWE = 3.0


def bosch_hale_dt_reactivity(T_keV):
    T = np.clip(np.asarray(T_keV, dtype=np.float64), 0.2, 100.0)
    BG = 34.3827
    mrc2 = 1.124656e6
    C1, C2, C3, C4 = 1.17302e-9, 1.51361e-2, 7.51886e-2, 4.60643e-3
    C5, C6, C7 = 1.35000e-2, -1.06750e-4, 1.36600e-5
    numerator = T * (C2 + T * (C4 + T * C6))
    denominator = 1.0 + T * (C3 + T * (C5 + T * C7))
    theta = T / (1.0 - numerator / denominator)
    xi = (BG**2 / (4.0 * theta)) ** (1.0 / 3.0)
    return 1.0e-6 * C1 * theta * np.sqrt(xi / (mrc2 * T**3)) * np.exp(-3.0 * xi)


def volume_rule(nodes=32):
    x, w = np.polynomial.legendre.leggauss(nodes)
    rho = 0.5 * (x + 1.0)
    weights = 0.5 * w * 2.0 * rho
    return rho.astype(np.float64), weights.astype(np.float64)


RHO, RHO_WEIGHTS = volume_rule(32)


def evaluate_cases(params):
    n_cases = len(next(iter(params.values())))
    gamma_n = params["gamma_n"]
    gamma_T = params["gamma_T"]
    total_ion_center = (N_D0_BASE + N_T0_BASE) * params["density_scale"]
    fuel_center = total_ion_center * params["fuel_fraction"]
    n_D0 = fuel_center * params["deuterium_fraction"]
    n_T0 = fuel_center * (1.0 - params["deuterium_fraction"])
    integral_reaction = np.zeros(n_cases)
    integral_pressure = np.zeros(n_cases)
    integral_brem_shape = np.zeros(n_cases)
    integral_n_total = np.zeros(n_cases)
    for rho, weight in zip(RHO, RHO_WEIGHTS):
        base = max(0.0, 1.0 - float(rho) ** 2)
        density_shape = base**gamma_n
        temperature_shape = base**gamma_T
        Ti = params["Ti_edge_keV"] + (params["Ti0_keV"] - params["Ti_edge_keV"]) * temperature_shape
        Te = np.maximum(0.2, Ti * params["Te_to_Ti"])
        n_i = total_ion_center * density_shape
        n_D = n_D0 * density_shape
        n_T = n_T0 * density_shape
        integral_reaction += weight * n_D * n_T * bosch_hale_dt_reactivity(Ti)
        integral_pressure += weight * (n_i * Ti + n_i * Te) * KEV_TO_J
        integral_brem_shape += weight * n_i**2 * np.sqrt(Te)
        integral_n_total += weight * n_i
    reaction_rate = CAD_PLASMA_VOLUME_M3 * integral_reaction
    fusion_W = reaction_rate * E_FUSION
    alpha_W = fusion_W * ALPHA_FRACTION
    neutron_W = fusion_W * NEUTRON_FRACTION
    alpha_deposited_W = alpha_W * params["alpha_deposition"]
    brem_W = CAD_PLASMA_VOLUME_M3 * 5.35e-37 * params["Z_eff"] * integral_brem_shape
    thermal_energy_J = 1.5 * CAD_PLASMA_VOLUME_M3 * integral_pressure
    transport_W = thermal_energy_J / params["tau_E_s"]
    laser_optical_W = RMAPLA_PULSE_ENERGY_J * params["repetition_rate_Hz"]
    laser_absorbed_W = laser_optical_W * params["laser_coupling"]
    self_ratio = alpha_deposited_W / np.maximum(brem_W + transport_W, 1.0)
    target_gain = (fusion_W / params["repetition_rate_Hz"]) / RMAPLA_PULSE_ENERGY_J
    beta = 2.0 * MU0 * integral_pressure / params["B0_T"]**2
    captured_W = neutron_W * params["neutron_heat_capture"] + alpha_W * params["alpha_heat_capture"] + laser_absorbed_W
    high_grade_W = captured_W * params["high_grade_heat_fraction"]
    low_grade_W = captured_W - high_grade_W
    gross_module_W = params["eta_high_grade"] * high_grade_W + params["eta_low_grade"] * low_grade_W
    gross_hub_GWe = gross_module_W * ACTIVE_MODULES / 1.0e9
    laser_electric_hub_GWe = laser_optical_W / params["wallplug_efficiency"] * ACTIVE_MODULES / 1.0e9
    recirc_hub_GWe = laser_electric_hub_GWe + params["nonlaser_aux_GWe"]
    net_hub_GWe = gross_hub_GWe - recirc_hub_GWe
    t_burn_g_h = reaction_rate * M_T * 1000.0 * 3600.0 * ACTIVE_MODULES
    t_surplus_g_h = params["tritium_process_eff"] * params["TBR"] * t_burn_g_h - t_burn_g_h
    tau_required = np.where(alpha_deposited_W > brem_W, thermal_energy_J / np.maximum(alpha_deposited_W - brem_W, 1.0), np.inf)
    tau_margin = params["tau_E_s"] / np.maximum(tau_required, 1.0e-30) - 1.0
    gate_all = (
        (fusion_W >= FUSION_SOURCE_TARGET_W_PER_MODULE)
        & (target_gain >= 200.0)
        & (params["wallplug_efficiency"] >= 0.15)
        & (self_ratio >= 1.0)
        & (params["TBR"] >= 1.15)
        & (t_surplus_g_h >= 0.0)
        & (captured_W >= CAPTURED_HEAT_TARGET_W_PER_MODULE)
        & (gross_hub_GWe >= GROSS_ELECTRIC_TARGET_GWE)
        & (recirc_hub_GWe <= RECIRCULATING_CEILING_GWE)
        & (net_hub_GWe >= NET_ELECTRIC_TARGET_GWE)
    )
    return pd.DataFrame({
        **{key: np.asarray(value) for key, value in params.items()},
        "fusion_GW_module": fusion_W / 1.0e9,
        "alpha_GW_module": alpha_W / 1.0e9,
        "neutron_GW_module": neutron_W / 1.0e9,
        "target_gain": target_gain,
        "self_heating_ratio": self_ratio,
        "beta_percent": 100.0 * beta,
        "thermal_energy_GJ_module": thermal_energy_J / 1.0e9,
        "brem_MW_module": brem_W / 1.0e6,
        "transport_GW_module": transport_W / 1.0e9,
        "captured_heat_GWth_module": captured_W / 1.0e9,
        "gross_GWe_hub": gross_hub_GWe,
        "laser_electric_GWe_hub": laser_electric_hub_GWe,
        "recirculating_GWe_hub": recirc_hub_GWe,
        "net_GWe_hub": net_hub_GWe,
        "T_burn_g_h_hub": t_burn_g_h,
        "T_surplus_g_h_hub": t_surplus_g_h,
        "tau_E_required_alpha_self_heating_s": tau_required,
        "tau_E_margin_ratio": tau_margin,
        "gate_all": gate_all,
    })


def reference_parameters(n=1):
    values = {
        "gamma_n": 0.657,
        "density_scale": 1.06,
        "fuel_fraction": 1.0,
        "deuterium_fraction": 0.5,
        "Ti0_keV": 20.0,
        "Ti_edge_keV": 1.25,
        "gamma_T": 0.75,
        "Te_to_Ti": 1.0,
        "B0_T": 14.0,
        "tau_E_s": 1.36,
        "Z_eff": 1.0,
        "alpha_deposition": 0.90,
        "laser_coupling": 0.80,
        "repetition_rate_Hz": 10.0,
        "wallplug_efficiency": 0.18,
        "neutron_heat_capture": 0.98,
        "alpha_heat_capture": 0.95,
        "high_grade_heat_fraction": 0.95,
        "eta_high_grade": 0.40,
        "eta_low_grade": 0.12,
        "nonlaser_aux_GWe": 1.333,
        "TBR": 1.18,
        "tritium_process_eff": 0.995,
    }
    return {key: np.full(n, value, dtype=np.float64) for key, value in values.items()}


REFERENCE = evaluate_cases(reference_parameters())
REFERENCE.to_csv(OUTPUT_ROOT / "reference_metrics.csv", index=False)
print(REFERENCE[["fusion_GW_module", "target_gain", "self_heating_ratio", "tau_E_required_alpha_self_heating_s", "tau_E_margin_ratio", "net_GWe_hub"]].to_string(index=False))


 fusion_GW_module  target_gain  self_heating_ratio  tau_E_required_alpha_self_heating_s  tau_E_margin_ratio  net_GWe_hub
         5.414417   216.576676            1.234227                             1.086268            0.251993    17.712282


## Deterministic proof-of-concept acceptance tests


In [5]:
# CELL 5 — Deterministic proof-of-concept acceptance tests
reference = REFERENCE.iloc[0]
test_results = {
    "geometry_all_checks": bool(GEOMETRY_CHECKS["all_geometry_checks_pass"]),
    "laser_energy_conservation": bool(pulse_checks["energy_conservation_pass"]),
    "laser_average_power_is_25_MW_per_module": bool(np.isclose(pulse_checks["module_average_power_MW"], 25.0)),
    "reference_fusion_is_finite_and_positive": bool(np.isfinite(reference["fusion_GW_module"]) and reference["fusion_GW_module"] > 0.0),
    "reference_self_heating_pass": bool(reference["self_heating_ratio"] >= 1.0),
    "reference_tau_E_margin_target_pass": bool(reference["tau_E_margin_ratio"] >= 0.20),
    "reference_net_export_is_finite": bool(np.isfinite(reference["net_GWe_hub"])),
}
test_results["poc_screening_pass"] = all(test_results.values())
test_results["physical_confinement_validation"] = "NOT_SOLVED_BY_THIS_NOTEBOOK"
test_results["interpretation"] = "Deterministic proof-of-concept consistency only; not 3-D MHD or reactor validation."
(OUTPUT_ROOT / "poc_checks.json").write_text(json.dumps(test_results, indent=2), encoding="utf-8")
print("PoC test results:", json.dumps(test_results, indent=2))
assert test_results["poc_screening_pass"]


PoC test results: {
  "geometry_all_checks": true,
  "laser_energy_conservation": true,
  "laser_average_power_is_25_MW_per_module": true,
  "reference_fusion_is_finite_and_positive": true,
  "reference_self_heating_pass": true,
  "reference_tau_E_margin_target_pass": true,
  "reference_net_export_is_finite": true,
  "poc_screening_pass": true,
  "physical_confinement_validation": "NOT_SOLVED_BY_THIS_NOTEBOOK",
  "interpretation": "Deterministic proof-of-concept consistency only; not 3-D MHD or reactor validation."
}


## Confinement-time robustness


In [6]:
# CELL 6 — Confinement-time robustness sweep
factors = np.array([0.90, 0.95, 1.00, 1.05, 1.10])
rows = []
for factor in factors:
    params = reference_parameters()
    params["tau_E_s"][:] = reference["tau_E_s"] * factor
    row = evaluate_cases(params).iloc[0]
    rows.append({
        "tau_E_factor": factor,
        "tau_E_s": row["tau_E_s"],
        "self_heating_ratio": row["self_heating_ratio"],
        "tau_E_margin_ratio": row["tau_E_margin_ratio"],
        "threshold_pass": bool(row["self_heating_ratio"] >= 1.0),
    })
tau_scan = pd.DataFrame(rows)
tau_scan.to_csv(OUTPUT_ROOT / "tau_E_robustness.csv", index=False)
fig, ax = plt.subplots(figsize=(8.0, 4.8))
ax.plot(tau_scan["tau_E_s"], tau_scan["self_heating_ratio"], marker="o", color="#6b46c1")
ax.axhline(1.0, color="#b52a2a", linestyle="--", label="self-heating threshold")
ax.set_xlabel("Energy-confinement time tau_E (s)")
ax.set_ylabel("P_alpha,deposited / modeled losses")
ax.set_title("R-MAPLA reference-point tau_E robustness")
ax.grid(True, alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "tau_E_robustness.png", dpi=160)
plt.close(fig)
print(tau_scan.to_string(index=False))


 tau_E_factor  tau_E_s  self_heating_ratio  tau_E_margin_ratio  threshold_pass
         0.90    1.224            1.118691            0.126794            True
         0.95    1.292            1.176663            0.189393            True
         1.00    1.360            1.234227            0.251993            True
         1.05    1.428            1.291386            0.314592            True
         1.10    1.496            1.348144            0.377192            True


## Smoke screen, then automatic full screen


In [7]:
# CELL 7 — Monte Carlo screening (smoke first, then full automatically)
PARAMETER_RANGES = {
    "gamma_n": (0.50, 1.00),
    "density_scale": (0.85, 1.20),
    "fuel_fraction": (0.80, 1.00),
    "deuterium_fraction": (0.45, 0.55),
    "Ti0_keV": (15.0, 25.0),
    "Ti_edge_keV": (0.20, 5.00),
    "gamma_T": (0.50, 2.00),
    "Te_to_Ti": (0.80, 1.20),
    "B0_T": (12.0, 16.0),
    "tau_E_s": (0.35, 2.00),
    "Z_eff": (1.00, 2.00),
    "alpha_deposition": (0.75, 0.98),
    "laser_coupling": (0.60, 0.90),
    "repetition_rate_Hz": (8.0, 12.0),
    "wallplug_efficiency": (0.10, 0.20),
    "neutron_heat_capture": (0.90, 0.99),
    "alpha_heat_capture": (0.85, 0.99),
    "high_grade_heat_fraction": (0.70, 0.98),
    "eta_high_grade": (0.36, 0.45),
    "eta_low_grade": (0.08, 0.20),
    "nonlaser_aux_GWe": (0.60, 1.60),
    "TBR": (0.88, 1.20),
    "tritium_process_eff": (0.970, 0.995),
}


def run_screen(n_cases, seed, label):
    start = time.monotonic()
    rng = np.random.default_rng(seed)
    batch_size = 4096
    parts = []
    for start_index in range(0, n_cases, batch_size):
        count = min(batch_size, n_cases - start_index)
        unit = rng.random((count, len(PARAMETER_RANGES)))
        params = {}
        for column, (name, bounds) in enumerate(PARAMETER_RANGES.items()):
            low, high = bounds
            params[name] = low + unit[:, column] * (high - low)
        frame = evaluate_cases(params)
        frame.insert(0, "sample_id", np.arange(start_index, start_index + count))
        parts.append(frame)
    result = pd.concat(parts, ignore_index=True)
    out_dir = OUTPUT_ROOT / label
    out_dir.mkdir(parents=True, exist_ok=True)
    result.nlargest(min(1000, len(result)), "net_GWe_hub").to_csv(out_dir / "top_net_cases.csv", index=False)
    pass_rate = float(result["gate_all"].mean() * 100.0)
    metrics = ["fusion_GW_module", "target_gain", "self_heating_ratio", "beta_percent", "net_GWe_hub", "tau_E_margin_ratio"]
    quantile_rows = []
    for metric in metrics:
        values = result[metric].replace([np.inf, -np.inf], np.nan).dropna()
        quantile_rows.append({"metric": metric, "P05": values.quantile(0.05), "P50": values.quantile(0.50), "P95": values.quantile(0.95), "mean": values.mean()})
    pd.DataFrame(quantile_rows).to_csv(out_dir / "quantiles.csv", index=False)
    summary = {
        "label": label,
        "cases": int(len(result)),
        "all_gate_passes": int(result["gate_all"].sum()),
        "all_gate_pass_percent": pass_rate,
        "elapsed_seconds": time.monotonic() - start,
        "best_net_GWe_hub": float(result["net_GWe_hub"].max()),
        "best_fusion_GW_module": float(result["fusion_GW_module"].max()),
        "best_self_heating_ratio": float(result["self_heating_ratio"].max()),
    }
    (out_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(label, json.dumps(summary, indent=2))
    return result, summary


SMOKE_RESULT, SMOKE_SUMMARY = run_screen(SMOKE_SAMPLES, SEED, "smoke")
print("Smoke screen complete; full screen starts automatically in the next cell.")


smoke {
  "label": "smoke",
  "cases": 8192,
  "all_gate_passes": 14,
  "all_gate_pass_percent": 0.1708984375,
  "elapsed_seconds": 0.10260233799999696,
  "best_net_GWe_hub": 35.47956266509517,
  "best_fusion_GW_module": 9.583433718097256,
  "best_self_heating_ratio": 2.022990022934545
}
Smoke screen complete; full screen starts automatically in the next cell.


In [8]:
# CELL 8 — Full proof-of-concept screen
FULL_RESULT, FULL_SUMMARY = run_screen(FULL_SAMPLES, SEED + 1, "full")
print("Full proof-of-concept screen complete.")


full {
  "label": "full",
  "cases": 262144,
  "all_gate_passes": 495,
  "all_gate_pass_percent": 0.1888275146484375,
  "elapsed_seconds": 1.184595105000028,
  "best_net_GWe_hub": 40.70463113350357,
  "best_fusion_GW_module": 11.268112881369932,
  "best_self_heating_ratio": 2.2070090925980153
}
Full proof-of-concept screen complete.


## Manifest and downloadable outputs


In [9]:
# CELL 9 — Final manifest and downloadable result bundle
manifest = {
    "model_revision": "2.2-RMAPLA-PoC",
    "status": "preliminary_proof_of_concept_screening_only",
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "geometry_checks": GEOMETRY_CHECKS,
    "pulse_checks": pulse_checks,
    "poc_checks": test_results,
    "smoke_summary": SMOKE_SUMMARY,
    "full_summary": FULL_SUMMARY,
    "physical_confinement_validation": "NOT_SOLVED",
    "notes": [
        "The 3-D geometry and energy/timing bookkeeping pass deterministic consistency checks.",
        "The fusion calculation is a dependency-free 0-D screening model.",
        "No 3-D equilibrium, closed-flux-surface solution, MHD stability spectrum, target hydrodynamics, neutronics, structural FEA, or experimental validation is performed.",
        "The symmetric ±5 cm focal-offset envelope is provisional and must be replaced by an optical design prescription.",
    ],
}
(OUTPUT_ROOT / "poc_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
zip_path = OUTPUT_ROOT / "Tomahawk_Rev22_RMAPLA_PoC_Results.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in sorted(OUTPUT_ROOT.rglob("*")):
        if path.is_file() and path != zip_path:
            archive.write(path, path.relative_to(OUTPUT_ROOT))
print("PoC manifest:", OUTPUT_ROOT / "poc_manifest.json")
print("Download bundle:", zip_path)


PoC manifest: /kaggle/working/tomahawk_rev22_poc/poc_manifest.json
Download bundle: /kaggle/working/tomahawk_rev22_poc/Tomahawk_Rev22_RMAPLA_PoC_Results.zip


## Interpretation limits

A passing result means the updated geometry, pulse bookkeeping, and simplified 0-D model are internally consistent for the selected assumptions. It does not establish plasma confinement, ignition, net electricity, tritium breeding, material lifetime, or engineering readiness. Treat the generated CSVs, plots, and manifest as preliminary inputs for higher-fidelity equilibrium, transport, neutronics, target, and hardware studies.
